# Attention: seminar of 9 November 2026

This executable edition follows the demonstrated attention material: dot-product scores, numerical stability and temperature, attention context, self-attention, permutation equivariance, and multiple heads.

**Post-seminar correction:** matrix softmax must normalize each row independently; its denominator retains the last axis (`keepdims=True`). The live coding version omitted this axis, so broadcasting could silently normalize columns when the score matrix was square. This edition fixes that issue and checks row sums and permutation equivariance.

The deliberately unstable softmax examples remain: warnings and non-finite values there illustrate overflow. The appended, unfinished general/concat attention and historical Colab/RNN templates are outside this executed lesson; the original notebook is preserved in the preparation archive.

Random matrices illustrate the operations; they are not trained model parameters. Run all cells in order. Dependencies: NumPy, Matplotlib, Seaborn.


## Practice: Attention Basics
Original notebook is provided by Udacity at [github](https://github.com/udacity/deep-learning-v2-pytorch/blob/master/attention/Attention_Basics.ipynb).

In this notebook, we look at how attention is implemented. We will focus on implementing attention in isolation from a larger model. That's because when implementing attention in a real-world model, a lot of the focus goes into piping the data and juggling the various vectors rather than the concepts of attention themselves.

We will implement attention scoring as well as calculating an attention context vector.

## Attention Scoring
### Inputs to the scoring function
Let's start by looking at the inputs we'll give to the scoring function. We will assume we're in the first step in the decoding phase. The first input to the scoring function is the hidden state of decoder (assuming a toy RNN with three hidden nodes -- not usable in real life, but easier to illustrate):

In [ ]:
dec_hidden_state = [5,1,20]

Let's visualize this vector:

In [ ]:
%matplotlib inline
import numpy as np
np.random.seed(20261109)
import matplotlib.pyplot as plt
import seaborn as sns

# Let's visualize our decoder hidden state
plt.figure(figsize=(1.5, 4.5))
sns.heatmap(np.transpose(np.matrix(dec_hidden_state)), annot=True, cmap=sns.light_palette("purple", as_cmap=True), linewidths=1)

Our first scoring function will score a single annotation (encoder hidden state), which looks like this:

In [ ]:
annotation = [3,12,45] #e.g. Encoder hidden state

In [ ]:
# Let's visualize the single annotation
plt.figure(figsize=(1.5, 4.5))
sns.heatmap(np.transpose(np.matrix(annotation)), annot=True, cmap=sns.light_palette("orange", as_cmap=True), linewidths=1)

### IMPLEMENT: Scoring a Single Annotation
Let's calculate the dot product of a single annotation. NumPy's [dot()](https://docs.scipy.org/doc/numpy/reference/generated/numpy.dot.html) is a good candidate for this operation

In [ ]:
def single_dot_attention_score(dec_hidden_state, enc_hidden_state):
    # TODO: return the dot product of the two vectors
    return np.asarray(dec_hidden_state).dot(enc_hidden_state)

single_dot_attention_score(dec_hidden_state, annotation)


### Annotations Matrix
Let's now look at scoring all the annotations at once. To do that, here's our annotation matrix:

In [ ]:
annotations = np.transpose([[3,12,45], [59,2,5], [1,43,5], [4,3,45.3]])

And it can be visualized like this (each column is a hidden state of an encoder time step):

In [ ]:
# Let's visualize our annotation (each column is an annotation)
annotations = np.transpose([[3,12,45], [59,2,5], [1,43,5], [4,3,45.3]])

ax = sns.heatmap(annotations, annot=True, cmap=sns.light_palette("orange", as_cmap=True), linewidths=1)

### IMPLEMENT: Scoring All Annotations at Once
Let's calculate the scores of all the annotations in one step using matrix multiplication. Let's continue to us the dot scoring method

<img src="https://github.com/neychev/made_nlp_course/blob/master/week05_machine_translation/img/scoring_functions.png?raw=1" />

To do that, we'll have to transpose `dec_hidden_state` and [matrix multiply](https://docs.scipy.org/doc/numpy/reference/generated/numpy.matmul.html) it with `annotations`.

In [ ]:
def dot_attention_score(dec_hidden_state, annotations):
    dec_as_array = np.asarray(dec_hidden_state)
    annotations_as_array = np.asarray(annotations)
    # TODO: return the product of dec_hidden_state transpose and enc_hidden_states
    return dec_as_array.dot(annotations_as_array)

attention_weights_raw = dot_attention_score(dec_hidden_state, annotations)
attention_weights_raw

In [ ]:
def unstable_softmax(x, T=1.):
  exps = np.exp(x)
  return exps/exps.sum()

In [ ]:
x = np.arange(4)

In [ ]:
plt.plot(np.arange(-100, 100), np.exp(np.arange(-100, 100)/15))

In [ ]:
T = 0.01
plt.bar(x, unstable_softmax(x+10000))

In [ ]:
np.allclose(unstable_softmax(x), unstable_softmax(x+2))

In [ ]:
plt.bar(x, unstable_softmax(x/20))

Looking at these scores, can you guess which of the four vectors will get the most attention from the decoder at this time step?

## Softmax
Now that we have our scores, let's apply softmax:
<img src="https://github.com/neychev/made_nlp_course/blob/master/week05_machine_translation/img/softmax.png?raw=1" />

In [ ]:
attention_weights_raw[:, None].shape

In [ ]:
def softmax(x, normalize=True):
    x = np.array(np.asarray(x).squeeze(), dtype=np.float32)
    if normalize:
        x -= x.max(axis=-1)
    e_x = np.exp(x)
    return e_x / e_x.sum(axis=-1, keepdims=True)

attention_weights = softmax(attention_weights_raw)
attention_weights

In [ ]:
np.array([10016], dtype=np.float16)

In [ ]:
for mult in [1, 10, 100, 1000, 10000, 100000]:
  print(f"mult: {mult}\n{softmax(x+mult, normalize=True)}\n{softmax(x+mult, normalize=False)}\n\n")

Even when knowing which annotation will get the most focus, it's interesting to see how drastic softmax makes the end score become. The first and last annotation had the respective scores of 927 and 929. But after softmax, the attention they'll get is 0.12 and 0.88 respectively.

# Applying the scores back on the annotations
Now that we have our scores, let's multiply each annotation by its score to proceed closer to the attention context vector. This is the multiplication part of this formula (we'll tackle the summation part in the latter cells)

<img src="https://github.com/neychev/made_nlp_course/blob/master/week05_machine_translation/img/Context_vector.png?raw=1" />

In [ ]:
def apply_attention_scores(attention_weights, annotations):

    # TODO: Multiple the annotations by their weights
    return annotations * attention_weights

applied_attention = apply_attention_scores(attention_weights, annotations)
applied_attention

Let's visualize how the context vector looks now that we've applied the attention scores back on it:

In [ ]:
attention_weights

In [ ]:
# Let's visualize our annotations after applying attention to them
ax = sns.heatmap(applied_attention, annot=True, cmap=sns.light_palette("orange", as_cmap=True), linewidths=1)

Contrast this with the raw annotations visualized earlier in the notebook, and we can see that the second and third annotations (columns) have been nearly wiped out. The first annotation maintains some of its value, and the fourth annotation is the most pronounced.

# Calculating the Attention Context Vector
All that remains to produce our attention context vector now is to sum up the four columns to produce a single attention context vector


In [ ]:
def calculate_attention_vector(applied_attention):
    return np.sum(applied_attention, axis=1)

attention_vector = calculate_attention_vector(applied_attention)
attention_vector

In [ ]:
# Let's visualize the attention context vector
plt.figure(figsize=(1.5, 4.5))
sns.heatmap(np.transpose(np.matrix(attention_vector)), annot=True, cmap=sns.light_palette("Blue", as_cmap=True), linewidths=1)

Now that we have the context vector, we can concatenate it with the hidden state and pass it through a hidden layer to produce the the result of this decoding time step.

In [ ]:
# Let's visualize our annotation (each column is an annotation)
x_sequence = np.random.randn(3, 4)

ax = sns.heatmap(x_sequence, annot=True, cmap=sns.light_palette("orange", as_cmap=True), linewidths=1)

In [ ]:
Wq = np.random.randn(3, 2)
Wk = np.random.randn(3, 2)
Wv = np.random.randn(3, 2)

In [ ]:
Wq

In [ ]:
query_vectors = x_sequence.T.dot(Wq)

In [ ]:
key_vectors = x_sequence.T.dot(Wk)

In [ ]:
value_vectors = x_sequence.T.dot(Wv)

In [ ]:
raw_attention_scores = query_vectors.dot(key_vectors.T) / np.sqrt(2)

In [ ]:
raw_attention_scores

In [ ]:
ax = sns.heatmap(raw_attention_scores, annot=True, cmap=sns.light_palette("red", as_cmap=True), linewidths=1)

In [ ]:
raw_attention_scores

In [ ]:
raw_attention_scores.max(axis=1)

In [ ]:
def softmax(x, normalize=True):
    x = np.array(x, dtype=np.float64)
    if normalize:
        x -= x.max(axis=-1, keepdims=True)
    e_x = np.exp(x)
    return e_x / e_x.sum(axis=-1, keepdims=True)

In [ ]:
attention_scores = softmax(raw_attention_scores, normalize=True)

In [ ]:
ax = sns.heatmap(attention_scores, annot=True, cmap=sns.light_palette("blue", as_cmap=True), linewidths=1)

In [ ]:
np.sum(value_vectors * attention_scores[1, :][:, None], axis=0)

In [ ]:
z_out_1 = attention_scores.dot(value_vectors).T

In [ ]:
z_out_1

In [ ]:
x_sequence.shape

In [ ]:
def compute_one_attention_head(x_sequence, Wq, Wk, Wv):
    query_vectors = x_sequence.T.dot(Wq)
    key_vectors = x_sequence.T.dot(Wk)
    value_vectors = x_sequence.T.dot(Wv)

    raw_attention_scores = query_vectors.dot(key_vectors.T) / np.sqrt(query_vectors.shape[-1])

    attention_scores = softmax(raw_attention_scores, normalize=True)
    z_out = attention_scores.dot(value_vectors).T

    return z_out

In [ ]:
compute_one_attention_head(x_sequence, Wq, Wk, Wv)

In [ ]:
x_sequence[:, ::-1]

In [ ]:
out_normal = compute_one_attention_head(x_sequence, Wq, Wk, Wv)

In [ ]:
out_inverced = compute_one_attention_head(x_sequence[:, ::-1], Wq, Wk, Wv)

In [ ]:
out_normal.shape

In [ ]:
out_inverced.shape

In [ ]:
np.allclose(out_normal, out_inverced[:, ::-1])

In [ ]:
def compute_multihead_attention(x_sequence, hidden_dim=2, number_of_heads=5):
  z_outs = []
  input_dim = x_sequence.shape[0]
  Wo = np.random.randn(input_dim, number_of_heads*hidden_dim)
  # print(Wo.shape)
  for head in range(number_of_heads):
      Wq = np.random.randn(input_dim, hidden_dim)
      Wk = np.random.randn(input_dim, hidden_dim)
      Wv = np.random.randn(input_dim, hidden_dim)

      z_outs.append(compute_one_attention_head(x_sequence, Wq, Wk, Wv))
  out = np.concatenate(z_outs, axis=0)
  # print(out.shape)
  return Wo.dot(out)

In [ ]:
x_sequence.shape # dim_size, seq_len


In [ ]:
out = compute_multihead_attention(x_sequence, 2, 5)
out2 = compute_multihead_attention(out, 4, 3)
out3 = compute_multihead_attention(out2, 11, 2)

for a in [out, out2, out3]:
    print(a.shape)

In [ ]:
out3[:, -1]

In [ ]:
x_sequence.shape, out.shape, out2.shape, out3.shape

In [ ]:
ax = sns.heatmap(out, annot=True, cmap=sns.light_palette("blue", as_cmap=True), linewidths=1)

## Checks for the implemented operation
The weights of every query sum to one. Reordering a sequence reorders its self-attention output in the same way, because no positional encoding has been added.


In [ ]:
scores = softmax(raw_attention_scores)
np.testing.assert_allclose(scores.sum(axis=-1), np.ones(scores.shape[0]))
np.testing.assert_allclose(out_normal, out_inverced[:, ::-1], rtol=1e-10, atol=1e-10)
# A rectangular matrix also needs row-wise normalization, not column broadcasting.
rectangular_scores = softmax(np.array([[1., 3., 2.], [-2., 0., 4.]]))
np.testing.assert_allclose(rectangular_scores.sum(axis=-1), np.ones(2))
np.testing.assert_allclose(softmax(raw_attention_scores + 10000), scores, rtol=1e-10, atol=1e-10)
assert np.isfinite(out3).all()
print("All attention checks passed.")
